# Рекомендация тарифов

В вашем распоряжении данные о поведении клиентов мобильного оператора. Нужно построить модель для задачи классификации, которая выберет подходящий тариф.

Постройте модель с максимально большим значением *accuracy*. Чтобы сдать проект успешно, нужно довести долю правильных ответов по крайней мере до 0.75. Проверьте *accuracy* на тестовой выборке самостоятельно.

Каждый объект в наборе данных — это информация о поведении одного пользователя за месяц. Известно:
* сalls — количество звонков,
* minutes — суммарная длительность звонков в минутах,
* messages — количество sms-сообщений,
* mb_used — израсходованный интернет-трафик в Мб,
* is_ultra — каким тарифом пользовался в течение месяца («Ультра» — 1, «Смарт» — 0).

**План работы:**
    
Шаг 1. Откроем файл с данными и изучим общую информацию
    
Шаг 2. Разделим исходные данные на обучающую, валидационную и тестовую выборки.
    
Шаг 3. Исследуем качество разных моделей, меняя гиперпараметры. Сформулируем выводы исследования.
    
Шаг 4. Проверим качество модели на тестовой выборке.
    
Шаг 5. Дополнительное задание: проверим модели на вменяемость. 

## Откройте и изучите файл

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from joblib import dump
import warnings
warnings.filterwarnings('ignore')

In [45]:
df = pd.read_csv('users_behavior.csv')# напишите в этой ячейке код для чтения таблицы 'df' из файла 'users_behavior.csv'


In [46]:
df.info()# напишите в этой ячейке код для отображения информации о таблице


<class 'pandas.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


In [47]:
df.head(5)# напишите в этой ячейке код для вывода первых 5 строк таблицы 'df'


,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


In [48]:
df['is_ultra'].value_counts()

is_ultra
0    2229
1     985
Name: count, dtype: int64

Определим признаки `features` и целевой признак `target `

In [49]:
features = df.drop(['is_ultra'], axis=1)
target = df['is_ultra']

**Вывод**

Каждая строка данных содержит информацию о поведении одного пользователя в течение месяца: 
* количество звонков, 
* суммарная длительность звонков в минутах, 
* количество sms-сообщений, 
* израсходованный интернет-трафик в Мб, 
* каким тарифом пользовался в течение месяца («Ультра» — 1, «Смарт» — 0).

Для построения модели определили признаки: 
* количество звонков, 
* суммарная длительность звонков в минутах, 
* количество sms-сообщений, 
* израсходованный интернет-трафик в Мб.

Выбрали целевой признак - каким тарифом пользовался в течение месяца («Ультра» — 1, «Смарт» — 0).

Количество абонентов тарифа "Ультра" в выборке меньше количества абонентов тарифа "Смарт" более, чем в 2 раза.

## Разбейте данные на выборки

Спрятанной тестовой выборки нет. Значит, данные нужно разбить на
три части: 
* обучающую, 
* валидационную и 
* тестовую. 

Размеры тестового и валидационного наборов обычно равны.

In [50]:
features_train, features_valid, target_train, target_valid = train_test_split(
   features, target, test_size=0.4, random_state=12345,stratify = target) 

In [51]:
features_valid, features_test, target_valid, target_test = train_test_split(features_valid, target_valid,  test_size=0.5, random_state=12345, stratify = target_valid)

**Вывод**

Исходные данные разбили в соотношении 3:1:1.

## Исследуйте модели

### Решающее дерево

In [64]:
best_model_tree = None
best_result_valid = 0
best_result_train = 0
# в цикле меняем высоту дерева не менее 5 раз: задайте диапазон в функции range(___,__) Должно получиться accuracy>=0.75
for depth in range(1,10):
    model = DecisionTreeClassifier(random_state=12345,max_depth=depth) # меняем высоту дерева
    model.fit(features_train, target_train)
    prediction_valid=model.predict(features_valid)
    prediction_train=model.predict(features_train)
    result_valid=accuracy_score(target_valid,prediction_valid)
    result_train=accuracy_score(target_train,prediction_train)
    print(f'Высота дерева = {depth}')
    print(f'Accuracy на обучающей выборке: {result_train}')
    print(f'Accuracy на валидационной выборке: {result_valid}')
    if result_train > best_result_train:
        best_depth_train=depth     # высота дерева для наилучшей модели
        best_model_tree_train = model   # наилучшая модель
        best_result_train = result_train # наилучшее значение метрики accuracy на обучающих данных
    if result_valid > best_result_valid:
        best_depth_valid=depth     # высота дерева для наилучшей модели
        best_model_tree_valid = model   # наилучшая модель
        best_result_valid = result_valid # наилучшее значение метрики accuracy на валидационных данных
print('---------------------------------------------------------------------------')
print(f'Accuracy наилучшей модели дерева на обучающей выборке: {best_result_train}')
print(f'Высота дерева для лучшей модели на обучающей выборке = {best_depth_train}')
print(f'Accuracy наилучшей модели дерева на валидационной выборке: {best_result_valid}')
print(f'Высота дерева для лучшей модели на валидационной выборке = {best_depth_valid}')

Высота дерева = 1
Accuracy на обучающей выборке: 0.7546680497925311
Accuracy на валидационной выборке: 0.7402799377916018
Высота дерева = 2
Accuracy на обучающей выборке: 0.7759336099585062
Accuracy на валидационной выборке: 0.7729393468118196
Высота дерева = 3
Accuracy на обучающей выборке: 0.7909751037344398
Accuracy на валидационной выборке: 0.7776049766718507
Высота дерева = 4
Accuracy на обучающей выборке: 0.7971991701244814
Accuracy на валидационной выборке: 0.7542768273716952
Высота дерева = 5
Accuracy на обучающей выборке: 0.8137966804979253
Accuracy на валидационной выборке: 0.7853810264385692
Высота дерева = 6
Accuracy на обучающей выборке: 0.8283195020746889
Accuracy на валидационной выборке: 0.7744945567651633
Высота дерева = 7
Accuracy на обучающей выборке: 0.8412863070539419
Accuracy на валидационной выборке: 0.7869362363919129
Высота дерева = 8
Accuracy на обучающей выборке: 0.8558091286307054
Accuracy на валидационной выборке: 0.80248833592535
Высота дерева = 9
Accuracy

**Вывод**

На валидационной выборке лучший результат модели:
* дерево с высотой ____ имеет accuracy модели _________.

## Проверьте модель на тестовой выборке

### Решающее дерево

В качестве модели для проверки на тестовой выборке возьмем модель решающего дерева, которая показала лучший результат на валидационной выборке.

Так как выборка у нас небольшая, попробуем улучшить модель на расширенной выборке (обучающая+валидационная):

In [82]:
# объединим обучающую и валидационную выборки
features_train_new = features_train._append(features_valid, ignore_index=True)
target_train_new = target_train._append(target_valid, ignore_index=True)

AttributeError: 'DataFrame' object has no attribute '_append'

In [59]:
# обучим модель на расширенной выборке
model = DecisionTreeClassifier(random_state=12345,max_depth=8)
model.fit(features_train_new, target_train_new)
result=model.score(features_train_new, target_train_new)
print(f'Accuracy на расширенной выборке: {result}')

Проверим модель на тестовой выборке:

In [31]:
answer = model.predict(features_test)

In [61]:
result_tree=accuracy_score(target_test,answer)
result_tree

In [63]:
print('Предсказания:',answer)
print('Правильные ответы:',target_test.values)

**Вывод**

На тестовой выборке модель решающего дерева показала  точность ___________